# Model Training and Comparison

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/heart_disease.csv')

In [3]:
X = df.drop(['Heart Disease Status'], axis=1)
y = df['Heart Disease Status']

In [4]:
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include=['str', 'object']).columns

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

num_pipeline = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, numeric_features),
    ("cat", cat_pipeline, categorical_features)
])

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

## Section 1: Baseline Models

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight='balanced'),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(random_state=42)
}

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, model in baseline_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

    results.append(scores)

baseline_results = pd.DataFrame(results)
baseline_results

d:\ML-Projects\Heart-Disease-Prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.5030,0.193182,0.4675,0.273392
1,KNN,0.7550,0.142857,0.0450,0.068441
2,Naive Bayes,0.8000,0.000000,0.0000,0.000000
3,SVM,0.5625,0.202753,0.4050,0.270225
4,Decision Tree,0.6740,0.197115,0.2050,0.200980
5,Random Forest,0.7980,0.357143,0.0125,0.024155
6,XGBoost,0.7710,0.097222,0.0175,0.029661


## Section 2: SMOTE (Synthetic Minority Oversampling Technique)

In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, model in baseline_models.items():
    smote_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

    smote_pipeline.fit(X_train, y_train)
    y_pred = smote_pipeline.predict(X_test)

    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

    results.append(scores)

smote_results = pd.DataFrame(results)
smote_results

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.5095,0.189968,0.4450,0.266268
1,KNN,0.4775,0.208145,0.5750,0.305648
2,Naive Bayes,0.5955,0.185868,0.3025,0.230257
3,SVM,0.7075,0.190635,0.1425,0.163090
4,Decision Tree,0.6595,0.197849,0.2300,0.212717
5,Random Forest,0.7985,0.000000,0.0000,0.000000
6,XGBoost,0.7705,0.152941,0.0325,0.053608


## Section 3: Hyperparameter Tuning + SMOTE

In [14]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline

tuned_results = []

params_grid = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },
    "KNN": {
        "model__n_neighbors": [2, 5, 10, 15, 20]
    },
    "SVM": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__kernel": ['linear', 'rbf', 'poly'],
        "model__gamma": ['scale', 'auto']
    },
    "Naive Bayes": {
    },
    "Decision Tree": {
        "model__criterion": ['entropy', 'log_loss'],
        "model__max_depth": [10, 50, 100],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200, 500],
        "model__max_depth": [5, 10, None],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2]
    },
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.1],
        "model__subsample": [0.8, 1.0]
    }
}

for name, model in baseline_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params_grid[name],
        scoring='f1',
        cv=10,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    tuned_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Best Params": grid_search.best_params_
    })

tuned_df = pd.DataFrame(tuned_results)
tuned_df

,Model,Accuracy,Precision,Recall,F1,Best Params
0,Logistic Regression,0.5095,0.189968,0.4450,0.266268,{'model__C': 0.01}
1,KNN,0.3890,0.200000,0.6850,0.309605,{'model__n_neighbors': 15}
2,Naive Bayes,0.5955,0.185868,0.3025,0.230257,{}
3,SVM,0.3490,0.196501,0.7300,0.309650,"{'model__C': 0.01, 'model__gamma': 'auto', 'mo..."
4,Decision Tree,0.6810,0.206897,0.2100,0.208437,"{'model__criterion': 'entropy', 'model__max_de..."
5,Random Forest,0.7985,0.200000,0.0025,0.004938,"{'model__max_depth': 5, 'model__min_samples_le..."
6,XGBoost,0.7905,0.148148,0.0100,0.018735,"{'model__learning_rate': 0.1, 'model__max_dept..."


## Observation

The dataset appears to contain weak relationships between predictors and the target variable, which may limit the achievable classification performance regardless of the algorithm used.